# AI 应用全栈开发面试八股


## 项目介绍与架构（A级）

### 1. 请介绍一下 GB-Agent

GB-Agent 是面向标准领域的本地智能代理平台。系统支持标准文档上传、解析、切块、向量检索、标准问答、条款审查和报告生成。后端使用 FastAPI，Agent 层使用 LangGraph，BGE 负责 Embedding，ChromaDB 负责语义检索，SQLite 保存业务数据，前端使用原生 JavaScript，并通过 SSE 展示执行计划、工具进度、回答和来源。

### 2. 项目的核心调用链是什么

文档入库：上传 → 解析 → 清洗 → 提取元数据 → 切块 → Embedding → SQLite、ChromaDB 和 FTS5。

用户问答：前端提问 → FastAPI → 意图识别 → 规划 → 工具执行 → 检索原文 → 合成回答 → SSE 返回。

### 3. 为什么使用 LangGraph

Agent 任务包含状态传递、条件分支、工具循环、多步骤执行和最终合成。LangGraph 能把这些步骤表示为显式节点和边，统一维护 AgentState，便于扩展、观察和测试。普通函数也能实现简单流程，但复杂后会出现状态和分支难以维护的问题。

### 4. 项目是真正的自主 Tool Calling 吗

当前主要是规则规划驱动：先识别意图和复杂度，再根据预定义模板生成线性计划或 DAG，编排器按照计划调用工具。这样稳定、可解释、便于测试，但灵活性有限。后续可以让 LLM 通过结构化输出生成计划，再使用工具白名单、参数校验、最大步数和人工确认控制风险。

### 5. 项目最大的不足是什么

主 RAG 只有向量召回，缺少统一混合检索、Reranker 和系统化评测；前端没有使用 React；规则规划的动态性有限；SQLite 更适合单机部署；模型文本流目前是生成完成后分块输出。改进优先级是先建立评测集，再做混合检索和重排，然后完成 React 与 Docker 重构。

## Python（A级）

### 1. Python 中可变对象和不可变对象有什么区别

可变对象可以在原地址修改内容，如 `list`、`dict`、`set`；不可变对象修改时会产生新对象，如 `int`、`str`、`tuple`。函数参数传递的是对象引用，因此修改传入列表的内容会影响调用方，重新绑定变量则不会。

### 2. 浅拷贝和深拷贝有什么区别

浅拷贝只复制最外层容器，内部嵌套对象仍共享；深拷贝递归复制嵌套对象。包含列表或字典的复杂状态如果需要完全隔离，应使用深拷贝，但成本更高。

### 3. 生成器是什么

生成器使用 `yield` 按需产生数据，不一次性把全部结果放入内存。适合处理大文件、流式数据和分页结果。每次迭代会从上次暂停的位置继续。

### 4. 装饰器是什么

装饰器接收函数并返回增强后的函数，用于在不改业务主体的情况下增加日志、权限、缓存、重试等横切能力。FastAPI 的路由声明就是装饰器的常见应用。

### 5. 上下文管理器有什么用

上下文管理器通过 `with` 保证资源在成功或异常时都能正确释放，例如文件、锁和数据库事务。它由 `__enter__/__exit__` 或 `contextlib` 实现。

### 6. `async/await` 是什么

它是协作式并发模型。协程遇到可等待的 I/O 时主动让出事件循环，使同一线程可以处理其他任务。它适合网络、数据库等等待型任务，不会自动加速 CPU 密集计算。

### 7. 线程、进程和协程如何选择

I/O 密集任务优先异步或线程；CPU 密集任务优先进程或外部计算服务；协程开销最小但要求调用链使用异步接口。GB-Agent 将同步文档解析、Embedding 和本地模型调用放入工作线程，避免阻塞 FastAPI 事件循环。

### 8. GIL 是什么

CPython 的 GIL 保证同一进程内同一时刻通常只有一个线程执行 Python 字节码，因此多线程不能有效加速纯 Python CPU 密集任务，但等待 I/O 时会释放执行机会。部分 C 扩展也可能释放 GIL。

### 9. 类型注解有什么价值

类型注解提高可读性和静态检查能力。在 FastAPI 中还用于请求校验、响应序列化和生成 OpenAPI 文档，但 Python 运行时默认不会自动强制所有类型。

### 10. 异常处理应注意什么

只捕获能够处理的异常，保留原始异常链和日志，不使用空的 `except` 吞掉问题；用户错误、依赖故障和程序缺陷应返回不同状态。清理资源优先使用 `finally` 或上下文管理器。

## FastAPI 与后端（A级）

### 1. FastAPI 为什么快

FastAPI 基于 Starlette 和 ASGI，支持异步 I/O；Pydantic 负责高效校验。它的“快”既包括运行性能，也包括类型驱动的开发效率。实际性能仍取决于数据库、模型调用和业务实现。

### 2. ASGI 和 WSGI 有什么区别

WSGI 主要面向同步 HTTP 请求；ASGI 支持异步、WebSocket 和长连接。FastAPI 是 ASGI 应用，通常由 Uvicorn 运行。

### 3. Pydantic 在 FastAPI 中做什么

Pydantic 根据类型注解解析、转换和校验请求数据，也能序列化响应并生成 JSON Schema。输入不满足模型时，FastAPI 通常返回 422。

### 4. RESTful API 的核心原则是什么

用资源名设计 URL，用 HTTP 方法表达操作，用状态码表达结果，并尽量保持无状态。`GET` 查询、`POST` 创建、`PUT` 整体替换、`PATCH` 局部更新、`DELETE` 删除。

### 5. 什么是幂等性

同一请求执行一次或多次，对服务端最终状态影响相同。GET、PUT、DELETE 通常应是幂等的，POST 通常不是。支付或任务创建等接口可以使用幂等键防止重复提交。

### 6. 为什么不能在异步路由里直接执行耗时同步代码

同步耗时任务会占住事件循环线程，导致其他请求无法及时处理。可以使用异步库、`asyncio.to_thread()`、线程池、进程池或任务队列。

### 7. 中间件和依赖注入有什么区别

中间件包围所有匹配请求，适合日志、CORS 和统一追踪；依赖注入按路由声明，适合认证、数据库会话和业务前置条件。

### 8. 常见 HTTP 状态码

- `200`：成功。
- `201`：资源创建成功。
- `204`：成功但无响应体。
- `400`：请求语义错误。
- `401`：未认证。
- `403`：已认证但无权限。
- `404`：资源不存在。
- `409`：资源冲突。
- `422`：数据校验失败。
- `429`：请求过多。
- `500`：服务端未处理异常。
- `503`：依赖或服务暂不可用。

### 9. SSE 和 WebSocket 如何选择

SSE 是服务器到浏览器的单向事件流，基于 HTTP，适合回答和任务进度推送；WebSocket 是全双工长连接，适合双方持续高频通信。GB-Agent 的交互是一次请求后持续接收结果，因此 SSE 足够且更简单。

### 10. 如何设计统一异常处理

业务层抛出明确异常，路由或全局处理器将其映射为状态码和稳定响应结构；内部日志记录堆栈，对外避免泄露路径、密钥和数据库信息。

## 网络与浏览器（A级）

### 1. 从输入 URL 到页面显示发生了什么

浏览器解析 URL，查询 DNS，建立 TCP 连接；HTTPS 还要完成 TLS 握手；发送 HTTP 请求，服务器处理并返回响应；浏览器解析 HTML、CSS 和 JavaScript，构建 DOM/CSSOM、布局和绘制。连接和缓存可能被复用。

### 2. TCP 为什么需要三次握手

三次握手让双方确认发送和接收能力，并同步初始序列号。两次无法让服务端确认客户端已经收到服务端的确认，也更难处理历史重复连接请求。

### 3. TCP 和 UDP 的区别

TCP 面向连接、可靠、有序，提供重传和流量控制；UDP 无连接、不保证到达和顺序，但开销小、延迟低。HTTP/1.1 和 HTTP/2 通常基于 TCP，HTTP/3 基于 QUIC/UDP。

### 4. HTTP 和 HTTPS 的区别

HTTPS 是 HTTP 运行在 TLS 上，提供加密、完整性和服务器身份认证。它不能保证服务器业务本身可信，但能防止传输过程被窃听和篡改。

### 5. GET 和 POST 有什么区别

区别主要是语义：GET 获取资源，应安全且幂等；POST 提交数据或创建资源，通常非幂等。参数放 URL 还是请求体只是常见形式，不是安全性的根本区别，敏感数据仍需要 HTTPS。

### 6. Cookie、Session、Token 的区别

Cookie 是浏览器保存并随请求发送的小数据；Session 通常把状态保存在服务端，Cookie 只保存会话标识；Token 将认证信息放在签名令牌中，服务端可以少存会话状态，但撤销和续期更复杂。

### 7. 什么是跨域和 CORS

浏览器同源策略限制不同协议、域名或端口的页面读取响应。CORS 由服务器通过响应头声明允许的来源、方法和请求头。CORS 是浏览器安全机制，不是服务器之间的访问控制。

### 8. HTTP 缓存有哪些

强缓存通过 `Cache-Control` 或 `Expires` 直接复用本地内容；协商缓存通过 `ETag/If-None-Match` 或 `Last-Modified/If-Modified-Since` 向服务器确认，未修改时返回 304。

## 数据库与存储（A级）

### 1. 什么是事务 ACID

原子性保证事务全部成功或全部回滚；一致性保证事务前后满足约束；隔离性控制并发事务互相影响；持久性保证提交后数据不会因普通故障丢失。

### 2. 数据库索引是什么

索引是额外的数据结构，用空间和写入成本换取查询速度。常见 B+ 树索引适合等值、范围和排序。索引并非越多越好，低选择性字段和频繁写入表需要权衡。

### 3. 为什么常用 B+ 树而不是普通二叉树

B+ 树分支多、树高低，能减少磁盘 I/O；数据集中在叶子节点并通过链表连接，适合范围扫描。

### 4. 什么情况下索引可能失效

对索引列做函数运算、隐式类型转换、使用不能利用前缀的模糊匹配、组合索引不满足最左前缀，或者优化器判断全表扫描成本更低。具体情况应查看执行计划。

### 5. SQLite 和 PostgreSQL 如何选择

SQLite 是嵌入式文件数据库，零运维，适合单机和本地工具；PostgreSQL 是独立数据库服务，支持更强并发、权限、事务、扩展和运维能力，适合多用户生产系统。GB-Agent 当前定位本地部署，因此采用 SQLite。

### 6. SQLite WAL 是什么

WAL 将修改先追加到日志文件，读取者可继续读取旧快照，从而改善读写并发。但 SQLite 仍然是单写者模型，不适合高并发写入。

### 7. SQLite 和 ChromaDB 分别存什么

SQLite 保存文档、原文块、用户、会话、记忆和任务状态等结构化业务数据；ChromaDB 保存 Embedding、文本副本和元数据，用于向量相似度检索。

### 8. 什么是数据库连接池

连接池复用已建立的数据库连接，避免每次请求创建连接的成本，并限制最大连接数量。连接使用后必须归还，事务失败时要回滚。

### 9. 什么是脏读、不可重复读和幻读

脏读是读到未提交数据；不可重复读是同一事务两次读取同一行得到不同结果；幻读是同一条件查询出现新增或消失的行。隔离级别越高，并发能力通常越低。

### 10. 多个存储系统双写有什么风险

如果 SQLite 写成功而 ChromaDB 写失败，会产生不一致。可通过状态机、幂等操作、补偿删除、重试、一致性扫描和重建索引降低风险。

## JavaScript 与前端（A级）

### 1. `var`、`let`、`const` 的区别

`var` 是函数作用域且存在变量提升；`let` 和 `const` 是块级作用域，并存在暂时性死区。`const` 不能重新绑定变量，但对象内部内容仍可修改。现代代码优先使用 `const`，需要重新赋值时使用 `let`。

### 2. `==` 和 `===` 的区别

`==` 会进行隐式类型转换，容易产生意外结果；`===` 同时比较类型和值。通常优先使用严格相等。

### 3. 闭包是什么

函数可以访问其定义时所在词法作用域中的变量，即使外层函数已经返回。闭包用于封装状态、回调和函数工厂，但不必要地持有大对象可能导致内存不能及时释放。

### 4. 事件循环是什么

JavaScript 主线程执行调用栈；异步操作完成后把回调放入任务队列。当前同步代码结束后，事件循环先清空微任务队列，再处理下一个宏任务。Promise 回调属于微任务，定时器属于宏任务。

### 5. Promise 和 async/await 的关系

`async/await` 是 Promise 的语法封装。`async` 函数总是返回 Promise，`await` 暂停当前协程式执行，但不会阻塞整个浏览器线程。错误可以通过 `try/catch` 处理。

### 6. 防抖和节流有什么区别

防抖是在连续触发停止一段时间后执行，适合搜索输入；节流是在固定时间窗口内最多执行一次，适合滚动和拖拽。

### 7. localStorage、sessionStorage 和 Cookie 的区别

localStorage 长期保存在浏览器；sessionStorage 仅当前标签会话；Cookie 容量较小且可自动随请求发送。敏感认证信息需要考虑 XSS、CSRF 和 HttpOnly，不能简单存入 localStorage。

### 8. React 的核心思想是什么（岗位补缺）

React 用组件描述 UI，界面由状态驱动。Props 用于父组件传入数据，State 管理组件内部变化，状态变化触发重新渲染。列表需要稳定 key，副作用使用 `useEffect`。

### 9. `useEffect` 常见问题是什么

依赖项遗漏会读取旧值，依赖不稳定会反复执行；订阅、定时器和请求需要在清理函数中释放或取消。不要用 Effect 处理能够在渲染期间直接计算的数据。

### 10. GB-Agent 的前端框架是什么

没有使用前端框架，而是 HTML、CSS 和模块化原生 JavaScript。通过 fetch 调用 FastAPI，通过 ReadableStream 解析 SSE，通过 AbortController 停止生成，通过 DOM API 更新页面。不要把它说成 React 项目。

## RAG、Embedding 与向量数据库（A级）

### 1. 什么是 RAG

RAG 在生成回答前从外部知识库检索相关内容，把检索结果作为上下文交给 LLM。它用于补充私有或最新知识、减少幻觉并提供来源，但效果取决于解析、切块、召回和提示词。

### 2. Embedding 是什么

Embedding 把文本映射为高维数值向量，使语义相近的文本在向量空间中距离更近。文档块和查询必须由兼容模型生成向量。更换模型或维度后通常需要重建索引。

### 3. 余弦相似度是什么

余弦相似度比较两个向量夹角，关注方向而非长度。向量归一化后，点积可直接反映余弦相似度。

### 4. 文档为什么要切块

整篇文档太长且主题混杂，不利于精确检索，也会占用模型上下文。块太小会丢失上下文，块太大会带入噪声，所以需要结合文档结构、模型上下文和评测结果确定。

### 5. Top-K 如何选择

K 太小可能漏召回，太大会引入噪声并增加上下文长度。应使用问题集比较不同 K 下的 Recall@K、回答正确率、忠实度和延迟，而不是凭感觉固定。

### 6. 向量检索和关键词检索的区别

向量检索擅长语义相似和同义表达，但对编号、型号和精确术语可能不稳定；关键词检索擅长精确匹配但不理解语义。生产 RAG 常组合两者。

### 7. 什么是混合检索

混合检索同时召回向量结果和关键词结果，再通过加权分数、RRF 等方式融合。GB-Agent 已建立 ChromaDB 和 FTS5，但主问答链路尚未实现统一融合排序，因此不能说已经完整实现混合检索。

### 8. Reranker 有什么作用

第一阶段召回追求不漏，Reranker 使用更精细的模型重新评估查询与候选文本的相关性，提高前几名质量，但会增加延迟。

### 9. 如何评估 RAG

检索层可看 Recall@K、Precision@K、MRR；生成层可看答案正确性、忠实度、引用准确性和人工评分；系统层还要看延迟、失败率和成本。评测必须基于固定问题集。

### 10. 如何减少幻觉

提高文档解析和召回质量；明确要求依据上下文回答；上下文不足时拒答；展示引用；对答案进行事实或引用校验；使用较低温度。RAG 只能降低幻觉，不能完全消除。

## LLM、Agent 与 LangGraph（A级）

### 1. LLM、RAG 和 Agent 的区别

LLM 负责理解和生成语言；RAG 在生成前提供外部知识；Agent 在目标驱动下进行规划、调用工具、观察结果和继续执行。三者可以组合，但不是同一概念。

### 2. 什么是 Tool Calling

Tool Calling 是模型按照工具名称和参数 Schema 输出结构化调用请求，应用执行真实工具后把结果返回模型。模型只负责选择和填参，真正操作由应用代码完成。参数必须校验，敏感操作需要权限或人工确认。

### 3. Agent 为什么可能死循环

模型可能重复调用失败工具、一直尝试补充信息或无法判断目标已完成。需要最大迭代次数、超时、重复调用检测、停止条件和预算控制。

### 4. Agent 如何处理工具失败

记录结构化错误，区分可重试和不可重试故障；设置超时与有限重试；必要时调用备用工具；最终回答明确指出失败，不让模型编造成功结果。

### 5. 什么是 Agent Memory

短期记忆通常是当前会话消息和状态；长期记忆是跨会话保存的用户偏好或重要事实。长期记忆需要控制写入条件、隐私、过期、冲突和召回噪声。

### 6. LangGraph 的 Node、Edge 和 State 是什么

Node 是处理步骤，Edge 决定执行顺序，Conditional Edge 根据状态选择分支，State 是节点共享的数据。Checkpointer 可以保存执行状态，用于多轮会话或恢复。

### 7. 线性计划和 DAG 有什么区别

线性计划严格按顺序执行；DAG 用依赖关系表示任务，没有依赖冲突的节点可以并行。DAG 适合多个独立检查后再汇总的任务，但调度、错误传播和状态管理更复杂。

### 8. Prompt 中 system、user、assistant 有什么作用

system 定义角色和高优先级行为，user 表达用户需求，assistant 保存模型历史回复。应用还可以加入检索上下文和工具结果，但要防止外部文档中的提示注入覆盖系统约束。

### 9. Temperature 有什么影响

温度越低，输出通常越稳定和确定；越高，随机性和多样性越大。标准问答和合规场景更适合低温，但温度不是准确性的保证。

### 10. 如何防范 Prompt Injection

把上传文档当作不可信数据；在提示中明确文档内容不能改变系统规则；限制工具和权限；校验工具参数；敏感操作要求人工确认；不要把密钥或内部提示词暴露给模型上下文。

## Git、Linux 与 Docker（A级）

### 1. Git merge 和 rebase 的区别

merge 保留分支历史并生成合并提交；rebase 把提交重新应用到新基点，历史更线性但会改写提交哈希。公共分支上的已推送历史不要随意 rebase。

### 2. `git revert` 和 `git reset` 的区别

revert 创建新提交来反向撤销旧提交，适合共享历史；reset 移动分支指针，可能改写或丢弃本地历史，使用前要确认影响范围。

### 3. Git 冲突如何处理

先理解双方改动意图，手动编辑冲突标记，运行测试，再 `git add` 并继续 merge 或 rebase。不能只机械选择一边。

### 4. Linux 如何查看进程、端口和日志

常用命令包括 `ps`、`top`、`ss -lntp`、`lsof -i`、`journalctl`、`tail -f`。排障顺序一般是进程是否存在、端口是否监听、日志报什么、依赖是否可访问。

### 5. Linux 文件权限是什么

权限分为用户、组和其他人的读、写、执行。目录的执行权限表示能够进入和访问目录项。常用 `chmod` 修改权限，`chown` 修改所有者。

### 6. Docker 镜像和容器的区别

镜像是只读模板，包含应用和依赖；容器是镜像的运行实例，具有可写层和运行状态。容器删除后可写层会丢失，持久数据应放在 Volume 或外部存储。

### 7. Dockerfile 常见指令

`FROM` 指定基础镜像，`WORKDIR` 设置工作目录，`COPY` 复制文件，`RUN` 构建时执行，`ENV` 设置环境变量，`EXPOSE` 声明端口，`CMD/ENTRYPOINT` 指定启动命令。

### 8. Docker Compose 有什么用

Compose 用一个配置声明多个服务、网络、环境变量和持久卷，适合同时启动前端、后端、数据库等开发或部署环境。

### 9. 容器访问宿主机的 `localhost` 是谁

容器内的 `localhost` 指容器自身，不是宿主机。容器之间应使用 Compose 服务名通信；访问宿主机需要平台提供的特殊地址或网络配置。

### 10. 如何排查容器启动失败

先看 `docker compose ps` 和日志，再检查启动命令、环境变量、端口冲突、文件权限、挂载路径、健康检查和依赖服务是否就绪。

## 测试、日志与故障排查（B级）

### 1. 单元测试、集成测试和端到端测试的区别

单元测试验证单个函数或类；集成测试验证模块之间的契约，如 API 与数据库；端到端测试从用户界面或外部入口验证完整流程。测试越接近端到端越真实，但更慢、更难定位失败。

### 2. Mock 什么时候使用

Mock 用于隔离慢、昂贵、不稳定或不可控的外部依赖，例如模型和网络 API。但过度 Mock 会使测试只验证实现细节，关键集成仍需要真实依赖测试。

### 3. 日志应该记录什么

记录时间、级别、模块、请求或任务 ID、关键阶段、耗时和错误堆栈。不要记录密码、Token、完整敏感文档或个人隐私。

### 4. 如何排查一个接口很慢

先分段测量路由、数据库、Embedding、检索和 LLM 耗时；检查是否阻塞事件循环、是否重复加载模型、查询是否走索引、上下文是否过大；最后再做缓存、并发或模型优化。

### 5. 如何讲 Bug 修复案例

使用 STAR：出现什么现象；你负责什么；如何通过日志、复现、最小化范围定位；根因是什么；如何修复；增加了什么测试防止回归；最终结果如何。不要只说“用 AI 找到了问题”。

## AI 编程工具（A级）

### 1. 如何使用 Codex 或 Claude Code

可以用于代码库检索、需求拆解、生成初稿、补充测试、定位错误和重构建议。使用前先提供明确任务和上下文；生成后必须阅读 diff、运行测试并验证边界情况。

### 2. 如何保证 AI 生成代码的质量

把任务拆小，限定修改范围；要求先阅读现有代码和测试；检查依赖与安全风险；运行格式化、静态检查和测试；审阅关键算法、SQL、权限和异常处理；使用 Git 保留可回退记录。

### 3. AI 编程工具有哪些风险

可能生成不存在的 API、引入安全漏洞、忽略项目约束、扩大修改范围或泄露敏感信息。开发者仍对结果负责，不能把“AI 生成的”当作正确性证明。

### 4. 面试时如何描述

> 我使用 Codex 和 Claude Code 辅助阅读代码、拆解任务、生成测试和定位问题，但会限制修改范围，逐段审查变更，并通过自动化测试和实际运行验证结果。AI 提高的是实现和检索效率，架构选择、边界判断和最终质量仍由我负责。

## 行为面试（A级）

### 1. 为什么应聘这个岗位

我的项目经历与岗位的 AI 应用全栈方向高度相关，已经实践过 Python、FastAPI、RAG、Agent 编排、文档处理和前端接口联调。我希望进入真实业务团队，补强前端框架、数据库和部署运维能力，并参与 AI 工具从需求到上线的完整流程。

### 2. 你的优势是什么

技术上有完整 AI 应用项目经验，能把模型、检索、后端和前端串起来；经历上海军陆战队服役和学生组织管理让我具备执行力、抗压能力和团队协调能力。回答后必须给一个具体案例。

### 3. 你的不足是什么

当前前端项目主要使用原生 JavaScript，对 React 的工程实践还不够；生产级 PostgreSQL 和容器部署经验也需要加强。我已经制定了用 React 重构 GB-Agent 核心页面并完成 Docker Compose 部署的计划。不要回答与岗位无关的伪缺点。

### 4. 遇到不会的问题怎么办

先澄清目标和边界，查看日志、文档和已有代码，构造最小复现，再逐层排除。必要时向团队说明已验证的信息、当前假设和需要协助的点，而不是长时间无反馈。

### 5. 如何处理意见冲突

先对齐目标和评价标准，把分歧转化为可验证的问题；用数据、原型或测试比较方案；无法统一时由明确的负责人决策，并记录原因。重点不是证明自己正确，而是推动任务完成。

### 6. 为什么应该录用你

我已有与岗位直接相关的 AI 应用全栈项目，能够快速进入 Python、FastAPI、RAG 和 Agent 开发工作；同时具备较强执行力和自学能力。对于 React、PostgreSQL 和部署方面的缺口，我能明确识别并通过项目实践快速补齐。

## 面试前必背 20 题

下面 20 题应做到每题 30～60 秒说清：

1. 一分钟介绍 GB-Agent。
2. 文档入库链路是什么？
3. 用户问答链路是什么？
4. 为什么使用 LangGraph？
5. 当前是否属于自主 Tool Calling？
6. RAG 是什么，为什么需要？
7. 文档为什么切块，如何切？
8. Embedding 和向量检索是什么？
9. SQLite、ChromaDB、FTS5 分别做什么？
10. 当前是否实现了混合检索？
11. 如何降低幻觉？
12. async/await 解决什么问题？
13. 为什么同步模型调用不能直接放在异步路由里？
14. SSE 和 WebSocket 有什么区别？
15. RESTful 和幂等性是什么？
16. 数据库索引和事务是什么？
17. 项目的前端框架是什么？
18. Docker 镜像、容器和 Volume 是什么？
19. 如何验证 AI 生成的代码？
20. 项目最大的不足和下一步计划是什么？

练习方法：随机抽题，先用一句话回答，再补充两点原理，最后用 GB-Agent 举例。连续三轮不看笔记说清，才算掌握。

## 不要说错的内容

- 不要说主问答链路已经完成向量与 FTS5 混合检索。
- 不要说 LLM 会从所有工具中完全自主规划；当前主要是规则模板规划。
- 不要说 SSE 文本是模型原生 token 流；当前是完成生成后分块输出。
- 不要说 ChromaDB 保存全部业务数据；业务数据主要在 SQLite。